# Module 8: Conditioning & Guidance

Conditioning is what transforms a diffusion model from a novelty into a controllable generative tool. This module covers the theory and implementation of **class-conditional generation**, **classifier guidance**, and **classifier-free guidance (CFG)** -- the technique used by virtually every state-of-the-art system today.

**Learning Objectives**
- Modify a UNet to accept class labels for conditional generation
- Implement classifier guidance using a separately trained noisy-image classifier
- Implement classifier-free guidance (CFG) with random label dropout during training
- Understand the guidance scale formula and its effect on the diversity-fidelity tradeoff
- Understand text conditioning via cross-attention at a conceptual level
- Understand how negative prompts work mechanically within the CFG framework

**Estimated time:** 3--4 hours

**Key references:**
- Classifier-Free Diffusion Guidance -- Ho & Salimans 2022: https://arxiv.org/abs/2207.12598
- Diffusion Models Beat GANs on Image Synthesis (ADM) -- Dhariwal & Nichol 2021: https://arxiv.org/abs/2105.05233
- Imagen -- Saharia et al. 2022: https://arxiv.org/abs/2205.11487

In [ ]:
import sys
import math
from typing import Optional, Dict, List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

sys.path.insert(0, '.')
from utils.schedule import cosine_schedule, get_schedule
from utils.visualization import show_images, denormalize, plot_loss_curve, show_denoising_trajectory, set_style
from utils.data import get_mnist_dataloader, get_device

torch.manual_seed(42)
device = get_device()
print(f"Using device: {device}")
set_style()
from torchvision.utils import make_grid


---
## 8.1 -- Class-Conditional Generation

An unconditional diffusion model learns to predict noise as $\epsilon_\theta(x_t, t)$. It generates images from the overall data distribution but gives us no control over *what* it generates. A **conditional** model instead learns $\epsilon_\theta(x_t, t, c)$, where $c$ is a conditioning signal -- in this section, a class label.

### How to inject a class label

There are three standard approaches:

| Method | Mechanism | Pros | Cons |
|--------|-----------|------|------|
| **Embedding + addition** | Map class to a learned vector, add to timestep embedding | Simple, effective, widely used | Limited expressiveness for complex conditions |
| **Concatenation** | One-hot encode class, concatenate to input channels | Easy to implement | Increases input dimensionality |
| **Adaptive Group Norm (AdaGN)** | Class embedding modulates scale/shift of GroupNorm layers | Most expressive (used in ADM) | More complex implementation |

We use **embedding + addition** here: the class label (an integer) is mapped through `nn.Embedding` to a learned vector, then added to the timestep embedding. This is the same mechanism the model already uses for timesteps -- clean and effective.

### Architecture overview

```
class_label (int)  -->  nn.Embedding  -->  class_emb (time_dim,)
timestep (int)     -->  SinusoidalEmb -->  time_emb  (time_dim,)
                                            |
                                     combined = time_emb + class_emb
                                            |
                                         MLP projection
                                            |
                                     injected into each ResBlock
```

### Building the Class-Conditional UNet

Below we define the full UNet with an optional `num_classes` parameter. When `num_classes` is provided, the model allocates an embedding table of size `num_classes + 1` -- the extra entry is the **null token** used for unconditional generation (critical for CFG later). The class embedding is added to the sinusoidal timestep embedding before the shared MLP projection.

In [ ]:
class SinusoidalTimestepEmbedding(nn.Module):
    """Sinusoidal positional embedding for diffusion timesteps.
    
    Maps integer timestep t -> vector of size embed_dim using
    sine/cosine frequencies, identical to the transformer positional encoding.
    """
    def __init__(self, embed_dim: int) -> None:
        super().__init__()
        self.embed_dim = embed_dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """Args: t -- (B,) integer timesteps. Returns: (B, embed_dim)."""
        half = self.embed_dim // 2
        freqs = torch.exp(
            -math.log(10000.0) * torch.arange(half, device=t.device, dtype=torch.float32) / half
        )  # (half,)
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)  # (B, half)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # (B, embed_dim)

In [ ]:
class ResBlock(nn.Module):
    """Residual block with time (and class) conditioning via addition.
    
    Architecture: GroupNorm -> SiLU -> Conv -> (+ time_emb) -> GroupNorm -> SiLU -> Dropout -> Conv -> (+ skip)
    """
    def __init__(self, in_channels: int, out_channels: int, time_dim: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.time_proj = nn.Linear(time_dim, out_channels)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.skip_conv = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        """Args: x -- (B,C,H,W), t_emb -- (B, time_dim). Returns: (B, out_C, H, W)."""
        h = self.conv1(F.silu(self.norm1(x)))       # (B, out_C, H, W)
        h = h + self.time_proj(F.silu(t_emb))[:, :, None, None]  # broadcast time emb
        h = self.conv2(self.dropout(F.silu(self.norm2(h))))  # (B, out_C, H, W)
        return h + self.skip_conv(x)                # (B, out_C, H, W)

In [ ]:
class AttentionBlock(nn.Module):
    """Single-head self-attention over spatial dimensions."""
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.norm = nn.GroupNorm(8, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.proj_out = nn.Conv2d(channels, channels, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        h = self.norm(x)
        qkv = self.qkv(h).reshape(B, 3, C, H * W)  # (B, 3, C, N)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]  # each (B, C, N)
        attn = torch.bmm(q.transpose(1, 2), k) * (C ** -0.5)  # (B, N, N)
        attn = attn.softmax(dim=-1)
        out = torch.bmm(v, attn.transpose(1, 2))    # (B, C, N)
        out = out.reshape(B, C, H, W)
        return x + self.proj_out(out)                # (B, C, H, W)

In [ ]:
class DownBlock(nn.Module):
    """Downsampling block: ResBlock (+ optional Attention) then 2x downsample."""
    def __init__(self, in_ch: int, out_ch: int, time_dim: int, has_attn: bool = False, dropout: float = 0.0) -> None:
        super().__init__()
        self.res = ResBlock(in_ch, out_ch, time_dim, dropout)
        self.attn = AttentionBlock(out_ch) if has_attn else nn.Identity()
        self.downsample = nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1)

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h = self.res(x, t_emb)
        h = self.attn(h)
        return self.downsample(h), h  # downsampled, skip connection


class UpBlock(nn.Module):
    """Upsampling block: upsample, concatenate skip, then ResBlock (+ optional Attention)."""
    def __init__(self, in_ch: int, out_ch: int, time_dim: int, has_attn: bool = False, dropout: float = 0.0) -> None:
        super().__init__()
        self.upsample = nn.ConvTranspose2d(in_ch, in_ch, 2, stride=2)
        self.res = ResBlock(in_ch + out_ch, out_ch, time_dim, dropout)  # concat skip doubles channels
        self.attn = AttentionBlock(out_ch) if has_attn else nn.Identity()

    def forward(self, x: torch.Tensor, skip: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        x = self.upsample(x)                          # (B, in_ch, 2H, 2W)
        x = torch.cat([x, skip], dim=1)               # (B, in_ch + out_ch, 2H, 2W)
        x = self.res(x, t_emb)
        return self.attn(x)

In [ ]:
class UNet(nn.Module):
    """UNet noise-prediction network with optional class conditioning.

    When num_classes is provided, the model learns an embedding table of size
    (num_classes + 1) -- the +1 reserves index `num_classes` as the null/unconditional
    token. The class embedding is added to the sinusoidal timestep embedding before
    the shared MLP projection, so the conditioning information flows into every
    ResBlock identically to timestep information.

    Args:
        image_channels: Number of input/output image channels (1 for MNIST).
        base_channels: Base channel count (doubled at each resolution level).
        channel_mults: Multipliers for base_channels at each level.
        num_classes: Number of classes for conditional generation. None = unconditional.
        dropout: Dropout rate in ResBlocks.
    """
    def __init__(
        self,
        image_channels: int = 1,
        base_channels: int = 64,
        channel_mults: Tuple[int, ...] = (1, 2, 4),
        num_classes: Optional[int] = None,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        time_dim = base_channels * 4
        self.num_classes = num_classes

        # --- Timestep embedding ---
        self.time_embed = SinusoidalTimestepEmbedding(time_dim)

        # --- Class embedding (optional) ---
        if num_classes is not None:
            # +1 for the null/unconditional class token
            self.class_embed = nn.Embedding(num_classes + 1, time_dim)
        else:
            self.class_embed = None

        # --- Shared MLP that projects combined embedding ---
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )

        # --- Encoder ---
        self.conv_in = nn.Conv2d(image_channels, base_channels, 3, padding=1)
        channels = [base_channels * m for m in channel_mults]
        self.downs = nn.ModuleList()
        in_ch = base_channels
        for i, out_ch in enumerate(channels):
            has_attn = (i == len(channels) - 1)  # attention at lowest resolution
            self.downs.append(DownBlock(in_ch, out_ch, time_dim, has_attn=has_attn, dropout=dropout))
            in_ch = out_ch

        # --- Bottleneck ---
        self.mid_res1 = ResBlock(in_ch, in_ch, time_dim, dropout)
        self.mid_attn = AttentionBlock(in_ch)
        self.mid_res2 = ResBlock(in_ch, in_ch, time_dim, dropout)

        # --- Decoder ---
        self.ups = nn.ModuleList()
        for i, out_ch in enumerate(reversed(channels)):
            has_attn = (i == 0)  # mirror encoder attention
            self.ups.append(UpBlock(in_ch, out_ch, time_dim, has_attn=has_attn, dropout=dropout))
            in_ch = out_ch

        # --- Output ---
        self.conv_out = nn.Sequential(
            nn.GroupNorm(8, base_channels),
            nn.SiLU(),
            nn.Conv2d(base_channels, image_channels, 3, padding=1),
        )

    def forward(
        self,
        x: torch.Tensor,
        t: torch.Tensor,
        class_label: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """Forward pass.

        Args:
            x: (B, C, H, W) noisy images.
            t: (B,) integer timesteps.
            class_label: (B,) integer class labels. Use num_classes for null token.

        Returns:
            (B, C, H, W) predicted noise.
        """
        # Build combined embedding
        emb = self.time_embed(t)                       # (B, time_dim)
        if self.class_embed is not None and class_label is not None:
            emb = emb + self.class_embed(class_label)  # (B, time_dim)
        emb = self.time_mlp(emb)                       # (B, time_dim)

        # Encoder
        x = self.conv_in(x)                            # (B, base_ch, H, W)
        skips = []
        for down in self.downs:
            x, skip = down(x, emb)
            skips.append(skip)

        # Bottleneck
        x = self.mid_res1(x, emb)
        x = self.mid_attn(x)
        x = self.mid_res2(x, emb)

        # Decoder
        for up, skip in zip(self.ups, reversed(skips)):
            x = up(x, skip, emb)

        return self.conv_out(x)                        # (B, img_ch, H, W)

In [ ]:
# Quick sanity check: verify shapes with and without class conditioning
torch.manual_seed(42)

# Unconditional
model_uncond = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4), num_classes=None)
x_test = torch.randn(2, 1, 28, 28)
t_test = torch.randint(0, 1000, (2,))
out_uncond = model_uncond(x_test, t_test)
print(f"Unconditional output shape: {out_uncond.shape}")  # (2, 1, 28, 28)

# Class-conditional (10 MNIST classes)
model_cond = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4), num_classes=10)
labels_test = torch.tensor([3, 7])
out_cond = model_cond(x_test, t_test, class_label=labels_test)
print(f"Conditional output shape:   {out_cond.shape}")    # (2, 1, 28, 28)

# Null token (unconditional pass through conditional model)
null_labels = torch.full((2,), 10)  # index 10 = null token for 10-class model
out_null = model_cond(x_test, t_test, class_label=null_labels)
print(f"Null-token output shape:    {out_null.shape}")    # (2, 1, 28, 28)

total_params = sum(p.numel() for p in model_cond.parameters())
print(f"\nConditional model parameters: {total_params:,}")

del model_uncond, model_cond  # free memory

### Exercise 8.1: Train a class-conditional model on MNIST

Using the `UNet` defined above with `num_classes=10`, write a standard DDPM training loop that:
1. Samples `(images, labels)` from the MNIST dataloader
2. Samples random timesteps and noise
3. Creates noisy images via the forward process
4. Passes `model(x_t, t, labels)` to predict noise
5. Computes MSE loss against the actual noise

Train for 3000 steps. This is a warm-up -- we will add CFG on top of this in later sections.

In [ ]:
# Exercise 8.1 -- YOUR CODE HERE
# Implement a class-conditional training loop.
# Hints:
#   - Use get_mnist_dataloader(batch_size=64)
#   - Use cosine_schedule(T=1000) for the noise schedule
#   - model = UNet(image_channels=1, base_channels=64, channel_mults=(1,2,4), num_classes=10).to(device)
#   - optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)
#   - Train for 3000 steps
#   - Pass labels directly to the model: model(x_t, t, class_label=labels)

pass

In [ ]:
# ✅ SOLUTION — try the exercise above before running this
torch.manual_seed(42)

# Setup
T = 1000
schedule = cosine_schedule(T)
sqrt_alphas_cumprod = schedule["sqrt_alphas_cumprod"].to(device)         # (T,)
sqrt_one_minus_alphas_cumprod = schedule["sqrt_one_minus_alphas_cumprod"].to(device)  # (T,)

dataloader = get_mnist_dataloader(batch_size=64)
model = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4), num_classes=10).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)

num_steps = 3000
losses = []
data_iter = iter(dataloader)

model.train()
for step in tqdm(range(num_steps), desc="Training (conditional)"):
    # Get batch (cycle through dataloader)
    try:
        images, labels = next(data_iter)
    except StopIteration:
        data_iter = iter(dataloader)
        images, labels = next(data_iter)

    images = images.to(device)  # (B, 1, 28, 28)
    labels = labels.to(device)  # (B,)
    B = images.shape[0]

    # Sample random timesteps
    t = torch.randint(0, T, (B,), device=device)  # (B,)

    # Sample noise and create noisy images
    noise = torch.randn_like(images)  # (B, 1, 28, 28)
    x_t = (
        sqrt_alphas_cumprod[t, None, None, None] * images
        + sqrt_one_minus_alphas_cumprod[t, None, None, None] * noise
    )  # (B, 1, 28, 28)

    # Predict noise (class-conditional)
    noise_pred = model(x_t, t, class_label=labels)  # (B, 1, 28, 28)

    # MSE loss
    loss = F.mse_loss(noise_pred, noise)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    losses.append(loss.item())

print(f"Final loss: {losses[-1]:.4f}")
plot_loss_curve(losses, title="Exercise 8.1: Class-Conditional Training Loss")

---
## 8.2 -- Classifier Guidance

Before CFG existed, [Dhariwal & Nichol (2021)](https://arxiv.org/abs/2105.05233) proposed **classifier guidance**: use a separately trained classifier $p_\phi(y \mid x_t)$ to steer the diffusion sampling process toward a desired class $y$.

### The algorithm

During sampling at each step $t$:

1. Predict noise with the (unconditional) diffusion model: $\hat{\epsilon} = \epsilon_\theta(x_t, t)$
2. Compute the classifier gradient: $g = \nabla_{x_t} \log p_\phi(y \mid x_t)$
3. Shift the noise prediction: $\tilde{\epsilon} = \hat{\epsilon} - s \cdot \sqrt{1 - \bar{\alpha}_t} \cdot g$
4. Use $\tilde{\epsilon}$ in the standard DDPM update step

Here $s$ is the **guidance scale**. Higher $s$ produces sharper, more class-consistent images at the cost of diversity.

### The problem

This requires training a **separate classifier on noisy images** (not clean images -- the classifier must handle arbitrary noise levels). This is wasteful and limits flexibility. This drawback motivates classifier-free guidance in Section 8.3.

### Worked Example: Noisy-Image Classifier

We train a small CNN classifier that takes noisy MNIST images (at various noise levels) and predicts the digit class. This classifier must be trained on noisy inputs because during sampling, $x_t$ is noisy.

In [ ]:
class NoisyClassifier(nn.Module):
    """Simple CNN classifier that operates on noisy images.
    
    Takes the noisy image and the timestep as input so it can
    account for the noise level.
    """
    def __init__(self, num_classes: int = 10, time_dim: int = 64) -> None:
        super().__init__()
        self.time_embed = SinusoidalTimestepEmbedding(time_dim)
        self.time_mlp = nn.Sequential(nn.Linear(time_dim, time_dim), nn.SiLU())

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),  # (B, 32, 28, 28)
            nn.SiLU(),
            nn.MaxPool2d(2),                  # (B, 32, 14, 14)
            nn.Conv2d(32, 64, 3, padding=1),  # (B, 64, 14, 14)
            nn.SiLU(),
            nn.MaxPool2d(2),                  # (B, 64, 7, 7)
            nn.Conv2d(64, 64, 3, padding=1),  # (B, 64, 7, 7)
            nn.SiLU(),
            nn.AdaptiveAvgPool2d(1),           # (B, 64, 1, 1)
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 + time_dim, 128),
            nn.SiLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """Args: x (B,1,28,28), t (B,). Returns: (B, num_classes) logits."""
        feat = self.features(x).squeeze(-1).squeeze(-1)  # (B, 64)
        t_emb = self.time_mlp(self.time_embed(t))        # (B, time_dim)
        combined = torch.cat([feat, t_emb], dim=1)       # (B, 64 + time_dim)
        return self.classifier(combined)                 # (B, num_classes)

In [ ]:
# Train the noisy-image classifier
torch.manual_seed(42)

classifier = NoisyClassifier(num_classes=10).to(device)
cls_optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)

cls_dataloader = get_mnist_dataloader(batch_size=128)
cls_data_iter = iter(cls_dataloader)
cls_losses = []

classifier.train()
for step in tqdm(range(2000), desc="Training classifier"):
    try:
        images, labels = next(cls_data_iter)
    except StopIteration:
        cls_data_iter = iter(cls_dataloader)
        images, labels = next(cls_data_iter)

    images, labels = images.to(device), labels.to(device)
    B = images.shape[0]

    # Add random noise (random timestep per sample)
    t = torch.randint(0, T, (B,), device=device)
    noise = torch.randn_like(images)
    x_t = (
        sqrt_alphas_cumprod[t, None, None, None] * images
        + sqrt_one_minus_alphas_cumprod[t, None, None, None] * noise
    )

    logits = classifier(x_t, t)
    loss = F.cross_entropy(logits, labels)

    cls_optimizer.zero_grad()
    loss.backward()
    cls_optimizer.step()
    cls_losses.append(loss.item())

print(f"Final classifier loss: {cls_losses[-1]:.4f}")
plot_loss_curve(cls_losses, title="Noisy-Image Classifier Training Loss")

### Classifier-Guided Sampling

Now we implement the sampling loop with classifier guidance. We need a trained unconditional diffusion model for this. Since we only trained a *conditional* model above, we will reuse it with the null class token to simulate unconditional predictions, then add the classifier gradient to steer sampling.

In [ ]:
@torch.no_grad()
def ddpm_sample_step(
    x_t: torch.Tensor,
    eps_pred: torch.Tensor,
    t_idx: int,
    schedule: Dict[str, torch.Tensor],
) -> torch.Tensor:
    """Single DDPM reverse step: x_{t-1} from x_t and predicted noise."""
    device = x_t.device
    beta_t = schedule["betas"][t_idx].to(device)
    alpha_t = schedule["alphas"][t_idx].to(device)
    alpha_bar_t = schedule["alphas_cumprod"][t_idx].to(device)
    sqrt_recip_alpha = schedule["sqrt_recip_alphas"][t_idx].to(device)

    # Predicted mean
    mean = sqrt_recip_alpha * (x_t - beta_t / (1.0 - alpha_bar_t).sqrt() * eps_pred)

    if t_idx == 0:
        return mean
    else:
        sigma = schedule["posterior_variance"][t_idx].to(device).sqrt()
        return mean + sigma * torch.randn_like(x_t)

In [ ]:
def classifier_guided_sample(
    model: UNet,
    classifier: NoisyClassifier,
    schedule: Dict[str, torch.Tensor],
    target_class: int,
    guidance_scale: float = 5.0,
    num_samples: int = 8,
    T: int = 1000,
    device: torch.device = torch.device("cpu"),
) -> torch.Tensor:
    """Generate samples using classifier guidance.

    Args:
        model: Trained diffusion model (used with null class token for unconditional prediction).
        classifier: Trained noisy-image classifier.
        schedule: Noise schedule dictionary.
        target_class: Class to guide toward.
        guidance_scale: Strength of classifier guidance (s).
        num_samples: Number of images to generate.
        T: Number of diffusion timesteps.
        device: Device.

    Returns:
        (num_samples, 1, 28, 28) generated images.
    """
    model.eval()
    classifier.eval()

    null_label = 10  # null class token for 10-class model
    x = torch.randn(num_samples, 1, 28, 28, device=device)  # (B, 1, 28, 28)

    for t_idx in reversed(range(T)):
        t_batch = torch.full((num_samples,), t_idx, device=device, dtype=torch.long)

        # Step 1: unconditional noise prediction (using null token)
        with torch.no_grad():
            null_labels = torch.full((num_samples,), null_label, device=device, dtype=torch.long)
            eps_pred = model(x, t_batch, class_label=null_labels)  # (B, 1, 28, 28)

        # Step 2: classifier gradient
        x_in = x.detach().requires_grad_(True)
        logits = classifier(x_in, t_batch)
        log_probs = F.log_softmax(logits, dim=-1)
        target = torch.full((num_samples,), target_class, device=device, dtype=torch.long)
        selected_log_prob = log_probs[range(num_samples), target].sum()
        grad = torch.autograd.grad(selected_log_prob, x_in)[0]  # (B, 1, 28, 28)

        # Step 3: shift noise prediction
        alpha_bar_t = schedule["alphas_cumprod"][t_idx].to(device)
        eps_guided = eps_pred - guidance_scale * (1.0 - alpha_bar_t).sqrt() * grad

        # Step 4: DDPM sampling step
        x = ddpm_sample_step(x, eps_guided, t_idx, schedule)

    return x

In [ ]:
# Generate digit 7 with different guidance scales
schedule_cpu = cosine_schedule(T)
target_digit = 7

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, s in zip(axes, [0.0, 2.0, 5.0, 10.0]):
    torch.manual_seed(42)
    samples = classifier_guided_sample(
        model, classifier, schedule_cpu,
        target_class=target_digit, guidance_scale=s,
        num_samples=4, T=T, device=device,
    )
    samples = denormalize(samples.cpu())
    grid = make_grid(samples, nrow=2, padding=1)
    ax.imshow(grid.permute(1, 2, 0).numpy(), cmap="gray")
    ax.set_title(f"s = {s}")
    ax.axis("off")
plt.suptitle(f"Classifier Guidance: Generating digit {target_digit}", fontsize=14)
plt.tight_layout()
plt.show()

### Exercise 8.2: Sweep classifier guidance scale

Generate digit 3 with guidance scales `s = [0, 1, 2, 5, 7, 10]`. Display a grid showing how the outputs change. What happens at very high guidance scales?

In [ ]:
# Exercise 8.2 -- YOUR CODE HERE
# Generate digit 3 at multiple guidance scales.
# Use classifier_guided_sample() with num_samples=4 for each scale.
# Plot a grid with one column per scale.

pass

In [ ]:
# ✅ SOLUTION — try the exercise above before running this
scales = [0, 1, 2, 5, 7, 10]
target_digit = 3

fig, axes = plt.subplots(1, len(scales), figsize=(3 * len(scales), 3))
for ax, s in zip(axes, scales):
    torch.manual_seed(42)
    samples = classifier_guided_sample(
        model, classifier, schedule_cpu,
        target_class=target_digit, guidance_scale=s,
        num_samples=4, T=T, device=device,
    )
    samples = denormalize(samples.cpu())
    grid = make_grid(samples, nrow=2, padding=1)
    ax.imshow(grid.permute(1, 2, 0).numpy(), cmap="gray")
    ax.set_title(f"s = {s}")
    ax.axis("off")
plt.suptitle(f"Classifier Guidance Sweep: Digit {target_digit}", fontsize=14)
plt.tight_layout()
plt.show()

print("At s=0 we get unconditioned (random digit) samples.")
print("As s increases, samples look more like the target digit but become less diverse.")
print("At very high s (e.g. 10), images can become oversaturated/distorted.")

---
## 8.3 -- Classifier-Free Guidance (CFG): The Key Insight

Classifier guidance has an obvious drawback: you need a separately trained classifier that works on noisy images. [Ho & Salimans (2022)](https://arxiv.org/abs/2207.12598) proposed an elegant alternative called **classifier-free guidance** that eliminates this requirement entirely.

### The brilliant trick

During **training**, randomly replace the class label with a **null token** with probability $p_{\text{uncond}}$ (typically 10--20%). This means the *same model* learns both:

- **Conditional** prediction: $\epsilon_\theta(x_t, t, c)$ -- when the real class label is provided
- **Unconditional** prediction: $\epsilon_\theta(x_t, t, \varnothing)$ -- when the null token is provided

No separate classifier needed. The model itself implicitly learns the difference between conditional and unconditional denoising.

### Implementation

The change to the training loop is minimal:

```python
# During training, randomly drop labels with probability p_uncond
mask = torch.rand(batch_size) < p_uncond      # which samples to make unconditional
labels[mask] = null_label                       # replace with null token (= num_classes)
noise_pred = model(x_t, t, class_label=labels)  # model sees a mix of real and null labels
```

That is it. The `nn.Embedding(num_classes + 1, time_dim)` we defined in Section 8.1 already reserves the null token at index `num_classes`.

### Worked Example: Training with label dropout

Let us retrain the model with 10% random label dropout -- the only change from Exercise 8.1.

In [ ]:
torch.manual_seed(42)

# Fresh model for CFG training
cfg_model = UNet(image_channels=1, base_channels=64, channel_mults=(1, 2, 4), num_classes=10).to(device)
cfg_optimizer = torch.optim.Adam(cfg_model.parameters(), lr=2e-4)

cfg_dataloader = get_mnist_dataloader(batch_size=64)
cfg_data_iter = iter(cfg_dataloader)
cfg_losses = []

p_uncond = 0.1     # probability of dropping the class label
null_label = 10    # null token index (num_classes)
num_steps = 5000

cfg_model.train()
for step in tqdm(range(num_steps), desc="Training with CFG label dropout"):
    try:
        images, labels = next(cfg_data_iter)
    except StopIteration:
        cfg_data_iter = iter(cfg_dataloader)
        images, labels = next(cfg_data_iter)

    images = images.to(device)  # (B, 1, 28, 28)
    labels = labels.to(device)  # (B,)
    B = images.shape[0]

    # THE KEY CHANGE: randomly drop class labels
    drop_mask = torch.rand(B, device=device) < p_uncond  # (B,)
    labels = labels.clone()
    labels[drop_mask] = null_label  # replace with null token

    # Standard DDPM forward process
    t = torch.randint(0, T, (B,), device=device)
    noise = torch.randn_like(images)
    x_t = (
        sqrt_alphas_cumprod[t, None, None, None] * images
        + sqrt_one_minus_alphas_cumprod[t, None, None, None] * noise
    )

    # Forward pass with (possibly dropped) labels
    noise_pred = cfg_model(x_t, t, class_label=labels)  # (B, 1, 28, 28)
    loss = F.mse_loss(noise_pred, noise)

    cfg_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(cfg_model.parameters(), 1.0)
    cfg_optimizer.step()

    cfg_losses.append(loss.item())

print(f"Final loss: {cfg_losses[-1]:.4f}")
plot_loss_curve(cfg_losses, title="CFG Training Loss (with 10% label dropout)")

In [ ]:
# Verify the same model produces different outputs for conditional vs unconditional
cfg_model.eval()
torch.manual_seed(0)

x_demo = torch.randn(1, 1, 28, 28, device=device)
t_demo = torch.tensor([500], device=device)

with torch.no_grad():
    # Conditional: predict noise for digit 7
    eps_cond = cfg_model(x_demo, t_demo, class_label=torch.tensor([7], device=device))
    # Unconditional: predict noise with null token
    eps_uncond = cfg_model(x_demo, t_demo, class_label=torch.tensor([null_label], device=device))

diff = (eps_cond - eps_uncond).abs().mean().item()
print(f"Mean absolute difference between conditional and unconditional predictions: {diff:.4f}")
print("This difference is what CFG amplifies during sampling.")

---
## 8.4 -- The Guidance Scale Formula

During sampling with CFG, we combine the conditional and unconditional noise predictions:

$$\tilde{\epsilon} = \epsilon_\theta(x_t, t, \varnothing) + s \cdot \big(\epsilon_\theta(x_t, t, c) - \epsilon_\theta(x_t, t, \varnothing)\big)$$

where $s$ is the **guidance scale**. This can be rewritten as:

$$\tilde{\epsilon} = (1 - s) \cdot \epsilon_{\text{uncond}} + s \cdot \epsilon_{\text{cond}}$$

### Interpreting the guidance scale

| Scale $s$ | Behavior | Interpretation |
|-----------|----------|----------------|
| $s = 0$ | Pure unconditional | Ignores class entirely |
| $s = 1$ | Standard conditional | Normal class-conditional output (no guidance amplification) |
| $s > 1$ | Amplified guidance | Sharper, more class-consistent (the sweet spot) |
| $s \gg 1$ | Over-guided | Saturated, low diversity, potential artifacts |
| $s < 0$ | Negative guidance | Steers *away* from the class |

Typical values: $s \in [2, 4]$ for class-conditional models, $s \in [7.5, 15]$ for text-to-image models.

### Mathematical interpretation

CFG implicitly samples from a distribution proportional to:

$$\tilde{p}(x \mid c) \propto p(x) \cdot p(c \mid x)^s$$

At $s = 1$ this is the true conditional $p(x \mid c)$. At $s > 1$, the conditioning signal is *amplified* -- the model generates images that are more "prototypical" of class $c$.

### Efficient implementation

CFG requires **two forward passes** per sampling step (one conditional, one unconditional). In practice, these are batched together:

```python
# Batch both passes into a single forward call
x_in = torch.cat([x_t, x_t], dim=0)           # (2B, C, H, W)
t_in = torch.cat([t, t], dim=0)                 # (2B,)
c_in = torch.cat([class_labels, null_labels])    # (2B,)
eps_both = model(x_in, t_in, class_label=c_in)   # (2B, C, H, W)
eps_cond, eps_uncond = eps_both.chunk(2)          # each (B, C, H, W)
eps_guided = eps_uncond + scale * (eps_cond - eps_uncond)
```

### Worked Example: CFG Sampling

We implement the full CFG sampling loop using the batched approach, then generate digits at various guidance scales.

In [ ]:
@torch.no_grad()
def cfg_sample(
    model: UNet,
    schedule: Dict[str, torch.Tensor],
    class_labels: torch.Tensor,
    guidance_scale: float = 3.0,
    T: int = 1000,
    device: torch.device = torch.device("cpu"),
) -> torch.Tensor:
    """Generate samples using classifier-free guidance.

    Args:
        model: UNet trained with label dropout (CFG-ready).
        schedule: Noise schedule dictionary.
        class_labels: (B,) desired class labels for each sample.
        guidance_scale: CFG scale (s). s=1 is standard conditional, s>1 is amplified.
        T: Number of diffusion timesteps.
        device: Device.

    Returns:
        (B, 1, 28, 28) generated images.
    """
    model.eval()
    B = class_labels.shape[0]
    null_label = model.num_classes  # null token index

    x = torch.randn(B, 1, 28, 28, device=device)  # (B, 1, 28, 28)

    for t_idx in reversed(range(T)):
        t_batch = torch.full((B,), t_idx, device=device, dtype=torch.long)  # (B,)

        # Batch conditional and unconditional forward passes
        x_in = torch.cat([x, x], dim=0)                                    # (2B, 1, 28, 28)
        t_in = torch.cat([t_batch, t_batch], dim=0)                        # (2B,)
        null_labels = torch.full((B,), null_label, device=device, dtype=torch.long)
        c_in = torch.cat([class_labels, null_labels], dim=0)               # (2B,)

        eps_both = model(x_in, t_in, class_label=c_in)                     # (2B, 1, 28, 28)
        eps_cond, eps_uncond = eps_both.chunk(2, dim=0)                     # each (B, 1, 28, 28)

        # CFG formula
        eps_guided = eps_uncond + guidance_scale * (eps_cond - eps_uncond)   # (B, 1, 28, 28)

        # DDPM reverse step
        x = ddpm_sample_step(x, eps_guided, t_idx, schedule)

    return x

In [ ]:
# Generate digit 7 at different guidance scales
target = 7
scales_to_show = [1.0, 2.0, 4.0, 8.0]

fig, axes = plt.subplots(1, len(scales_to_show), figsize=(4 * len(scales_to_show), 4))
for ax, s in zip(axes, scales_to_show):
    torch.manual_seed(42)
    labels_gen = torch.full((4,), target, device=device, dtype=torch.long)
    samples = cfg_sample(cfg_model, schedule_cpu, labels_gen, guidance_scale=s, T=T, device=device)
    samples = denormalize(samples.cpu())
    grid = make_grid(samples, nrow=2, padding=1)
    ax.imshow(grid.permute(1, 2, 0).numpy(), cmap="gray")
    ax.set_title(f"s = {s}", fontsize=13)
    ax.axis("off")
plt.suptitle(f"CFG Sampling: Digit {target} at Different Guidance Scales", fontsize=14)
plt.tight_layout()
plt.show()

### Exercise 8.3: CFG sampling with configurable scale

1. Generate 8 samples of digit 5 at guidance scales `s = [0, 0.5, 1, 2, 4, 8]`.
2. Display a grid showing quality/diversity tradeoff.
3. Verify that `s=1` matches pure conditional generation (the model with no guidance amplification).

In [ ]:
# Exercise 8.3 -- YOUR CODE HERE
# Use cfg_sample() with the cfg_model.
# Generate 8 samples of digit 5 at each scale.
# Display side by side.

pass

In [ ]:
# ✅ SOLUTION — try the exercise above before running this
target = 5
scales = [0.0, 0.5, 1.0, 2.0, 4.0, 8.0]

fig, axes = plt.subplots(1, len(scales), figsize=(3.5 * len(scales), 4))
for ax, s in zip(axes, scales):
    torch.manual_seed(42)
    labels_gen = torch.full((8,), target, device=device, dtype=torch.long)
    samples = cfg_sample(cfg_model, schedule_cpu, labels_gen, guidance_scale=s, T=T, device=device)
    samples = denormalize(samples.cpu())
    grid = make_grid(samples, nrow=4, padding=1)
    ax.imshow(grid.permute(1, 2, 0).numpy(), cmap="gray")
    ax.set_title(f"s = {s}", fontsize=12)
    ax.axis("off")
plt.suptitle(f"CFG Scale Sweep: Digit {target}", fontsize=14)
plt.tight_layout()
plt.show()

# Verify s=1 matches pure conditional (no guidance amplification)
# At s=1: eps_guided = eps_uncond + 1*(eps_cond - eps_uncond) = eps_cond
# So the guidance formula reduces to pure conditional prediction.
print("At s=1, the CFG formula simplifies to:")
print("  eps_guided = eps_uncond + 1.0 * (eps_cond - eps_uncond) = eps_cond")
print("This is exactly pure conditional generation -- no amplification.")

---
## 8.5 -- Why CFG Works: Trading Diversity for Fidelity

CFG exposes a fundamental tradeoff:

- At $s = 1$: standard conditional -- **diverse** outputs, but some may be ambiguous or low-quality.
- At $s > 1$: amplified conditioning -- **higher fidelity** (images clearly match the class), but reduced diversity (outputs cluster around the mode).
- At very high $s$: over-guided -- samples can become saturated or distorted.

This is analogous to **temperature** in language models. Lower temperature (high guidance) produces more predictable, "safe" outputs. Higher temperature (low guidance) produces more varied but less reliable outputs.

Every state-of-the-art text-to-image system (Stable Diffusion, DALL-E, Imagen, Midjourney) uses CFG with $s > 1$. The guidance scale is often one of the most important hyperparameters for output quality.

### Quantifying the tradeoff

We can measure this by generating many samples and computing:
- **Diversity**: average pairwise distance between samples (higher = more diverse)
- **Consistency**: how strongly samples match the target class (proxy for fidelity)

In [ ]:
def compute_diversity(samples: torch.Tensor) -> float:
    """Compute average pairwise L2 distance between samples.
    
    Args:
        samples: (B, C, H, W) tensor of generated images.
    Returns:
        Mean pairwise distance (scalar).
    """
    B = samples.shape[0]
    flat = samples.reshape(B, -1)  # (B, C*H*W)
    # Pairwise L2 distances
    dists = torch.cdist(flat, flat, p=2)  # (B, B)
    # Take upper triangle (exclude self-comparisons)
    mask = torch.triu(torch.ones(B, B, dtype=torch.bool), diagonal=1)
    return dists[mask].mean().item()

In [ ]:
# Measure diversity at different guidance scales
scales_sweep = [0.0, 0.5, 1.0, 2.0, 4.0, 6.0, 8.0]
diversities = []
num_diversity_samples = 16  # keep small for CPU feasibility

for s in tqdm(scales_sweep, desc="Diversity sweep"):
    torch.manual_seed(123)
    labels_gen = torch.full((num_diversity_samples,), 7, device=device, dtype=torch.long)
    samples = cfg_sample(
        cfg_model, schedule_cpu, labels_gen,
        guidance_scale=s, T=T, device=device,
    )
    diversities.append(compute_diversity(samples.cpu()))

# Plot
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(scales_sweep, diversities, 'o-', linewidth=2, markersize=8, color='steelblue')
ax.set_xlabel("Guidance Scale (s)", fontsize=12)
ax.set_ylabel("Diversity (avg pairwise L2)", fontsize=12)
ax.set_title("Diversity vs. Guidance Scale", fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("As guidance scale increases, diversity decreases -- samples converge toward the mode.")

---
## 8.6 -- Text Conditioning Preview

Class labels are just integers -- they carry minimal information. Modern text-to-image models like [Imagen (Saharia et al. 2022)](https://arxiv.org/abs/2205.11487) and Stable Diffusion condition on *text embeddings*, which are far richer.

### The text-to-image pipeline

```
"a photograph of a cat on a beach"
        |
   Text Encoder (CLIP or T5)
        |
   text_embeddings: (seq_len, embed_dim)
        |
   Cross-Attention in UNet
   Q = image features, K = text, V = text
        |
   Image features attend to text tokens
```

### Cross-attention mechanism

In each attention block of the UNet, **cross-attention** layers are added:
- **Q** (query) comes from the image feature maps
- **K** (key) and **V** (value) come from the text embeddings
- This allows each spatial location in the image to attend to relevant words in the prompt

| Component | Class Conditioning | Text Conditioning |
|-----------|-------------------|-------------------|
| Input | Integer class label | Token sequence |
| Embedding | `nn.Embedding(num_classes, dim)` | Pretrained text encoder (CLIP/T5) |
| Injection | Addition to time embedding | Cross-attention (Q from image, K/V from text) |
| Expressiveness | 1 of N classes | Free-form natural language |
| CFG dropout | Replace label with null token | Replace text with empty string "" |

### CFG with text

The mechanism is identical to what we implemented with class labels: during training, the text prompt is randomly replaced with an empty string (the null condition). During sampling, the CFG formula applies exactly the same way:

$$\tilde{\epsilon} = \epsilon_\theta(x_t, t, \varnothing) + s \cdot \big(\epsilon_\theta(x_t, t, \text{text}) - \epsilon_\theta(x_t, t, \varnothing)\big)$$

We will not implement text conditioning from scratch here (it requires a pretrained text encoder), but the conceptual framework is exactly what we built in Sections 8.3--8.4 -- just with richer embeddings and cross-attention instead of addition.

---
## 8.7 -- Negative Prompts: How They Work Mechanically

Negative prompts are a popular feature in text-to-image systems. They are not a separate mechanism -- they fall directly out of the CFG formula.

### The standard CFG formula

$$\tilde{\epsilon} = \epsilon_{\text{uncond}} + s \cdot (\epsilon_{\text{cond}} - \epsilon_{\text{uncond}})$$

### Negative prompt modification

Replace $\epsilon_{\text{uncond}}$ with $\epsilon_{\text{neg}}$ -- the noise prediction conditioned on the *negative* prompt:

$$\tilde{\epsilon} = \epsilon_{\text{neg}} + s \cdot (\epsilon_{\text{cond}} - \epsilon_{\text{neg}})$$

This steers the generation *away from* the negative prompt and *toward* the positive prompt. The larger $s$ is, the stronger both effects.

### Example in text-to-image

- Positive prompt: "a photograph of a cat"
- Negative prompt: "blurry, low quality"
- Result: the model generates a cat while actively avoiding blurriness

### With class labels

We can demonstrate the same idea with class labels: use one class as the positive target and another as the negative. The model will generate images that look like the positive class while avoiding the negative class.

In [ ]:
@torch.no_grad()
def cfg_sample_with_negative(
    model: UNet,
    schedule: Dict[str, torch.Tensor],
    class_labels: torch.Tensor,
    negative_labels: torch.Tensor,
    guidance_scale: float = 3.0,
    T: int = 1000,
    device: torch.device = torch.device("cpu"),
) -> torch.Tensor:
    """CFG sampling with negative prompt (class) support.

    Instead of using the unconditional prediction as the baseline,
    uses the negative class prediction. This steers away from the
    negative class and toward the positive class.

    Args:
        model: CFG-trained UNet.
        schedule: Noise schedule dictionary.
        class_labels: (B,) positive class labels.
        negative_labels: (B,) negative class labels (steer away from these).
        guidance_scale: CFG scale.
        T: Timesteps.
        device: Device.

    Returns:
        (B, 1, 28, 28) generated images.
    """
    model.eval()
    B = class_labels.shape[0]

    x = torch.randn(B, 1, 28, 28, device=device)

    for t_idx in reversed(range(T)):
        t_batch = torch.full((B,), t_idx, device=device, dtype=torch.long)

        # Batch positive and negative forward passes
        x_in = torch.cat([x, x], dim=0)                              # (2B, 1, 28, 28)
        t_in = torch.cat([t_batch, t_batch], dim=0)                  # (2B,)
        c_in = torch.cat([class_labels, negative_labels], dim=0)     # (2B,)

        eps_both = model(x_in, t_in, class_label=c_in)               # (2B, 1, 28, 28)
        eps_cond, eps_neg = eps_both.chunk(2, dim=0)                  # each (B, 1, 28, 28)

        # Negative prompt CFG formula
        eps_guided = eps_neg + guidance_scale * (eps_cond - eps_neg)  # (B, 1, 28, 28)

        x = ddpm_sample_step(x, eps_guided, t_idx, schedule)

    return x

### Brief Exercise: Negative prompts with class labels

Generate digit 1 with digit 7 as the negative prompt. Compare against standard CFG (where the negative is the null token). You should see that negative prompts steer the output away from the negative class.

In [ ]:
# Exercise -- YOUR CODE HERE
# 1. Generate digit 1 with standard CFG (null token as negative)
# 2. Generate digit 1 with digit 7 as negative prompt
# 3. Display side by side

pass

In [ ]:
# ✅ SOLUTION — try the exercise above before running this
torch.manual_seed(42)
B = 8
pos_labels = torch.full((B,), 1, device=device, dtype=torch.long)
null_labels_neg = torch.full((B,), null_label, device=device, dtype=torch.long)
neg_7_labels = torch.full((B,), 7, device=device, dtype=torch.long)

# Standard CFG (null as negative)
torch.manual_seed(42)
samples_standard = cfg_sample(
    cfg_model, schedule_cpu, pos_labels,
    guidance_scale=4.0, T=T, device=device,
)

# Negative prompt: steer away from 7
torch.manual_seed(42)
samples_neg = cfg_sample_with_negative(
    cfg_model, schedule_cpu, pos_labels, neg_7_labels,
    guidance_scale=4.0, T=T, device=device,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
grid_std = make_grid(denormalize(samples_standard.cpu()), nrow=4, padding=1)
grid_neg = make_grid(denormalize(samples_neg.cpu()), nrow=4, padding=1)
axes[0].imshow(grid_std.permute(1, 2, 0).numpy(), cmap="gray")
axes[0].set_title("Standard CFG (null negative)", fontsize=12)
axes[0].axis("off")
axes[1].imshow(grid_neg.permute(1, 2, 0).numpy(), cmap="gray")
axes[1].set_title("Negative prompt: digit 7", fontsize=12)
axes[1].axis("off")
plt.suptitle("Positive: digit 1 | Guidance scale = 4.0", fontsize=14)
plt.tight_layout()
plt.show()

print("With the negative prompt, the model steers away from digit-7-like features")
print("(e.g., avoiding diagonal strokes that 7s tend to have).")

---
## Capstone Exercise: Full Class-Conditional MNIST Diffusion with CFG

Bring everything together in a complete pipeline:

1. Build a `UNet` with `num_classes=10` (null token at index 10)
2. Train with 10% random label dropout for 8000 steps
3. Implement CFG-guided sampling
4. Generate a grid: **rows** = digits 0--9, **columns** = guidance scales 1, 2, 4, 8
5. Generate "the digit 7" at different scales
6. Compare unconditional ($s=0$) vs. guided ($s=4$)

This is a complete, interview-ready implementation of classifier-free guidance.

In [ ]:
# Capstone Exercise -- YOUR CODE HERE
#
# Follow the steps outlined above. You can reuse UNet, cfg_sample,
# cosine_schedule, and other utilities defined earlier.

pass

In [ ]:
# ✅ SOLUTION — try the exercise above before running this -- Part 1: Training
torch.manual_seed(42)

# Setup
T_cap = 1000
schedule_cap = cosine_schedule(T_cap)
sqrt_ac = schedule_cap["sqrt_alphas_cumprod"].to(device)
sqrt_omac = schedule_cap["sqrt_one_minus_alphas_cumprod"].to(device)

cap_dataloader = get_mnist_dataloader(batch_size=64)
cap_model = UNet(
    image_channels=1,
    base_channels=64,
    channel_mults=(1, 2, 4),
    num_classes=10,
).to(device)
cap_optimizer = torch.optim.Adam(cap_model.parameters(), lr=2e-4)

p_uncond_cap = 0.1  # 10% label dropout
null_label_cap = 10
num_steps_cap = 8000
cap_losses = []
cap_data_iter = iter(cap_dataloader)

cap_model.train()
for step in tqdm(range(num_steps_cap), desc="Capstone Training"):
    try:
        images, labels = next(cap_data_iter)
    except StopIteration:
        cap_data_iter = iter(cap_dataloader)
        images, labels = next(cap_data_iter)

    images = images.to(device)  # (B, 1, 28, 28)
    labels = labels.to(device)  # (B,)
    B = images.shape[0]

    # Random label dropout for CFG
    drop_mask = torch.rand(B, device=device) < p_uncond_cap
    labels = labels.clone()
    labels[drop_mask] = null_label_cap

    # Forward process
    t = torch.randint(0, T_cap, (B,), device=device)
    noise = torch.randn_like(images)
    x_t = sqrt_ac[t, None, None, None] * images + sqrt_omac[t, None, None, None] * noise

    # Noise prediction and loss
    noise_pred = cap_model(x_t, t, class_label=labels)
    loss = F.mse_loss(noise_pred, noise)

    cap_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(cap_model.parameters(), 1.0)
    cap_optimizer.step()

    cap_losses.append(loss.item())

print(f"Final loss: {cap_losses[-1]:.4f}")
plot_loss_curve(cap_losses, title="Capstone: CFG Training Loss (8000 steps)")

In [ ]:
# ✅ SOLUTION — try the exercise above before running this -- Part 2: Grid of digits x guidance scales
# Rows = digits 0-9, Columns = guidance scales 1, 2, 4, 8

cap_scales = [1.0, 2.0, 4.0, 8.0]
num_digits = 10

fig, axes = plt.subplots(num_digits, len(cap_scales), figsize=(3 * len(cap_scales), 2 * num_digits))

for row, digit in enumerate(range(num_digits)):
    for col, s in enumerate(cap_scales):
        torch.manual_seed(42)
        labels_gen = torch.full((1,), digit, device=device, dtype=torch.long)
        sample = cfg_sample(
            cap_model, schedule_cap, labels_gen,
            guidance_scale=s, T=T_cap, device=device,
        )
        img = denormalize(sample[0].cpu()).squeeze(0).numpy()  # (28, 28)
        axes[row, col].imshow(img, cmap="gray")
        axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(f"s = {s}", fontsize=12)
    axes[row, 0].set_ylabel(f"Digit {digit}", fontsize=11, rotation=0, labelpad=40)

plt.suptitle("Capstone: Digits 0-9 at Different CFG Scales", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ✅ SOLUTION — try the exercise above before running this -- Part 3: "Generate the digit 7" at different scales

scales_7 = [0.0, 1.0, 2.0, 4.0, 8.0]
fig, axes = plt.subplots(1, len(scales_7), figsize=(3.5 * len(scales_7), 4))

for ax, s in zip(axes, scales_7):
    torch.manual_seed(42)
    labels_gen = torch.full((8,), 7, device=device, dtype=torch.long)
    samples = cfg_sample(
        cap_model, schedule_cap, labels_gen,
        guidance_scale=s, T=T_cap, device=device,
    )
    samples = denormalize(samples.cpu())
    grid = make_grid(samples, nrow=4, padding=1)
    ax.imshow(grid.permute(1, 2, 0).numpy(), cmap="gray")
    ax.set_title(f"s = {s}", fontsize=13)
    ax.axis("off")
plt.suptitle("Generate Digit 7 at Different CFG Scales", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ✅ SOLUTION — try the exercise above before running this -- Part 4: Unconditional (s=0) vs Guided (s=4)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Unconditional: s=0 (class label is irrelevant since guidance is zero)
torch.manual_seed(42)
labels_gen = torch.full((16,), 7, device=device, dtype=torch.long)
samples_uncond = cfg_sample(
    cap_model, schedule_cap, labels_gen,
    guidance_scale=0.0, T=T_cap, device=device,
)
grid_uncond = make_grid(denormalize(samples_uncond.cpu()), nrow=8, padding=1)
axes[0].imshow(grid_uncond.permute(1, 2, 0).numpy(), cmap="gray")
axes[0].set_title("Unconditional (s = 0)", fontsize=13)
axes[0].axis("off")

# Guided: s=4
torch.manual_seed(42)
samples_guided = cfg_sample(
    cap_model, schedule_cap, labels_gen,
    guidance_scale=4.0, T=T_cap, device=device,
)
grid_guided = make_grid(denormalize(samples_guided.cpu()), nrow=8, padding=1)
axes[1].imshow(grid_guided.permute(1, 2, 0).numpy(), cmap="gray")
axes[1].set_title("Guided: digit 7, s = 4", fontsize=13)
axes[1].axis("off")

plt.suptitle("Unconditional vs. CFG-Guided Generation", fontsize=14)
plt.tight_layout()
plt.show()

print("At s=0, the model ignores the class label and generates random digits.")
print("At s=4, the model consistently generates the target digit with high fidelity.")

---
## Summary

This module covered the key techniques for controlling diffusion model outputs:

| Concept | Key Idea | Interview Takeaway |
|---------|----------|--------------------|
| **Class conditioning** | Add class embedding to timestep embedding | Simple injection via addition to existing MLP path |
| **Classifier guidance** | Use external classifier gradient to steer sampling | Requires separate noisy-image classifier (impractical) |
| **Classifier-free guidance** | Drop labels randomly during training; same model does both conditional and unconditional | The dominant approach -- no extra classifier needed |
| **Guidance scale** | $\tilde{\epsilon} = \epsilon_{\text{uncond}} + s(\epsilon_{\text{cond}} - \epsilon_{\text{uncond}})$ | s=1 is standard; s>1 trades diversity for fidelity |
| **Text conditioning** | Cross-attention with text encoder embeddings | Same CFG framework, richer conditioning signal |
| **Negative prompts** | Replace $\epsilon_{\text{uncond}}$ with $\epsilon_{\text{neg}}$ in CFG formula | Steers away from undesired features |

**Interview essentials:**
- CFG is used in every major text-to-image system (Stable Diffusion, DALL-E 3, Imagen, Midjourney)
- The training change is minimal: randomly drop labels with ~10% probability
- The sampling change requires two forward passes per step (or one batched pass)
- Guidance scale is one of the most impactful hyperparameters for output quality
- Mathematically, CFG samples from $p(x) \cdot p(c|x)^s$, amplifying the conditional signal